# Lab: Python, NumPy and Vectorization for Economic Analysis
A hands-on introduction to NumPy and vectorized computing, applied to core problems in economics: consumer expenditure, price indices, demand estimation, and multi-country panel data.

# Outline
- [1.1 Goals](#toc_1.1)
- [1.2 Useful References](#toc_1.2)
- [2 Python and NumPy for Economic Analysis](#toc_2)
- [3 Vectors in Economics](#toc_3)
  - [3.1 Abstract](#toc_3.1)
  - [3.2 NumPy Arrays](#toc_3.2)
  - [3.3 Vector Creation](#toc_3.3)
  - [3.4 Operations on Vectors](#toc_3.4)
    - [3.4.1 Indexing](#toc_3.4.1)
    - [3.4.2 Slicing](#toc_3.4.2)
    - [3.4.3 Single Vector Operations](#toc_3.4.3)
    - [3.4.4 Vector-Vector Element-wise Operations](#toc_3.4.4)
    - [3.4.5 Scalar-Vector Operations](#toc_3.4.5)
    - [3.4.6 Vector-Vector Dot Product](#toc_3.4.6)
    - [3.4.7 The Need for Speed: vector vs. for loop](#toc_3.4.7)
    - [3.4.8 Vector Operations in Demand Estimation](#toc_3.4.8)
- [4 Matrices in Economics: Panel Data & Input-Output Tables](#toc_4)
  - [4.1 Abstract](#toc_4.1)
  - [4.2 NumPy Arrays](#toc_4.2)
  - [4.3 Matrix Creation](#toc_4.3)
  - [4.4 Operations on Matrices](#toc_4.4)
    - [4.4.1 Indexing](#toc_4.4.1)
    - [4.4.2 Slicing](#toc_4.4.2)
  - [4.5 Bonus: Matrix-Vector Products in Input-Output Analysis](#toc_4.5)
- [5 Congratulations!](#toc_5)

In [ ]:
import numpy as np    # unofficial standard alias for NumPy
import time
import matplotlib.pyplot as plt

<a name="toc_1.1"></a>
## 1.1 Goals
In this lab, you will:
- Review the features of NumPy and Python used across quantitative economics and econometrics
- Represent economic data (prices, quantities, GDP, panel data) as NumPy vectors and matrices
- Use vectorization to compute expenditure, revenue, price indices, and demand predictions
- Confirm, empirically, why vectorized code is preferred over explicit `for` loops when working with economic datasets, which are often large (household surveys, high-frequency trading data, panel data across many countries and years)

<a name="toc_1.2"></a>
## 1.2 Useful References
- NumPy Documentation: [NumPy.org](https://NumPy.org/doc/stable/)
- Broadcasting (a key feature for economic data manipulation): [NumPy Broadcasting](https://NumPy.org/doc/stable/user/basics.broadcasting.html)
- `numpy.linalg` for solving linear economic models (e.g. Leontief input-output systems): [NumPy Linear Algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)

<a name="toc_2"></a>
# 2 Python and NumPy for Economic Analysis
Python is widely used across empirical economics, econometrics, and quantitative finance. NumPy extends Python with a fast, memory-efficient array type and a large library of numerical routines. Almost every economic dataset — a time series of GDP, a cross-section of household incomes, a panel of countries observed over many years, an input-output table of a national economy — can be represented naturally as a NumPy vector or matrix. Once represented this way, operations that would otherwise require explicit loops (computing total expenditure, adjusting for inflation, aggregating sector output) can be *vectorized*: expressed as a single, fast, array-level operation.

<a name="toc_3"></a>
# 3 Vectors in Economics
<a name="toc_3.1"></a>
## 3.1 Abstract
A great deal of microeconomic data is naturally a **vector**: an ordered list of numbers of the same type. For example:
- A **price vector** $\mathbf{p}$, where $p_i$ is the price of good $i$ in a consumer's basket
- A **quantity vector** $\mathbf{q}$, where $q_i$ is the quantity purchased of good $i$
- A **time series**, where $x_t$ is the value of GDP, inflation, or unemployment in period $t$

As in the general case, indexing in NumPy runs from 0 to $n-1$, so the price of the first good is $p_0$, not $p_1$.

<a name="toc_3.2"></a>
## 3.2 NumPy Arrays
NumPy's basic data structure is an indexable, n-dimensional *array* containing elements of the same type (`dtype`). A one-dimensional array (1-D) has one index and is exactly how we will represent a vector of prices, quantities, or a time series of an economic indicator.

 - 1-D array, shape (n,): n elements indexed [0] through [n-1]

<a name="toc_3.3"></a>
## 3.3 Vector Creation
Data creation routines in NumPy generally take a shape as their first parameter. Below, we create some placeholder economic vectors before filling them with real data.

In [ ]:
# NumPy routines that allocate memory and fill with a default value
a = np.zeros(5);                print(f"np.zeros(5)                : a = {a}, shape = {a.shape}, dtype = {a.dtype}")
a = np.zeros((5,));             print(f"np.zeros((5,))              : a = {a}, shape = {a.shape}, dtype = {a.dtype}")
a = np.random.random_sample(5); print(f"np.random.random_sample(5) : a = {a}, shape = {a.shape}, dtype = {a.dtype}")

Some routines do not take a shape tuple — useful for building an index of periods or goods:

In [ ]:
a = np.arange(5);   print(f"np.arange(5):      a = {a}, shape = {a.shape}, dtype = {a.dtype}  # good index 0..4")
a = np.random.rand(5); print(f"np.random.rand(5): a = {a}, shape = {a.shape}, dtype = {a.dtype}")

Now let's create the actual economic data we will use throughout this lab: prices and quantities for a small consumer basket of five goods.

In [ ]:
goods = ['Wheat', 'Oil', 'Steel', 'Electronics', 'Services']

p = np.array([200., 350., 500., 800., 150.])   # price per unit, $
q = np.array([120.,  80.,  60.,  40., 300.])   # quantity purchased, thousands of units

print(f"p (prices)    : {p}, shape = {p.shape}, dtype = {p.dtype}")
print(f"q (quantities): {q}, shape = {q.shape}, dtype = {q.dtype}")

<a name="toc_3.4"></a>
## 3.4 Operations on Vectors
<a name="toc_3.4.1"></a>
### 3.4.1 Indexing
**Indexing** means referring to *an element* of an array by its position. NumPy starts indexing at zero, so the price of the 3rd good, Steel, is `p[2]`.

In [ ]:
# access the price of Steel (3rd good, index 2)
print(f"Price of {goods[2]}: p[2] = {p[2]}")

# access the last good's quantity using a negative index
print(f"Quantity of last good ({goods[-1]}): q[-1] = {q[-1]}")

# indexes must be within range or NumPy raises an error
try:
    c = p[10]
except Exception as e:
    print("The error message you'll see is:")
    print(e)

<a name="toc_3.4.2"></a>
### 3.4.2 Slicing
Slicing creates a subset of an array using `start:stop:step`. This is useful, for example, to isolate a subset of goods (say, only manufactured goods) from a larger basket.

In [ ]:
print(f"p          = {p}")

# Oil, Steel, Electronics (indices 1 through 3)
c = p[1:4:1];  print("p[1:4:1] =", c, "->", goods[1:4])

# every other good starting at index 0
c = p[0:5:2];  print("p[0:5:2] =", c)

# all goods from Steel onward
c = p[2:];     print("p[2:]    =", c)

# all goods up to (not including) Steel
c = p[:2];     print("p[:2]    =", c)

# every good
c = p[:];      print("p[:]     =", c)

<a name="toc_3.4.3"></a>
### 3.4.3 Single Vector Operations
A number of useful economic quantities come from operations on a single vector.

In [ ]:
print(f"q                 : {q}")

# total units sold across all goods
total_q = np.sum(q)
print(f"total_q = np.sum(q)  : {total_q}  (thousand units)")

# average price across the basket
avg_p = np.mean(p)
print(f"avg_p = np.mean(p)   : {avg_p}  ($)")

# a simple quadratic cost term, e.g. cost(p) = p^2 (increasing marginal cost)
cost_term = p**2
print(f"cost_term = p**2     : {cost_term}")

# negate a vector, e.g. to represent a shortage instead of a surplus
shortage = -q
print(f"shortage = -q        : {shortage}")

<a name="toc_3.4.4"></a>
### 3.4.4 Vector-Vector Element-wise Operations
Most NumPy arithmetic, logical, and comparison operators work element-by-element. For example, the revenue earned from each good is $r_i = p_i \cdot q_i$, and the price change between two periods is $\Delta p_i = p^{new}_i - p^{old}_i$.

In [ ]:
# revenue by good, element-wise (NOT the total — see the dot product below for that)
revenue_by_good = p * q
print(f"revenue by good = p * q : {revenue_by_good}")

# prices one year later
p_next_year = np.array([210., 340., 520., 760., 155.])
price_change = p_next_year - p
print(f"price_change = p_next_year - p : {price_change}")

As with any element-wise operator, the vectors must be the same length:

In [ ]:
# mismatched vector operation
p_wrong_size = np.array([200., 350.])
try:
    d = p + p_wrong_size
except Exception as e:
    print("The error message you'll see is:")
    print(e)

<a name="toc_3.4.5"></a>
### 3.4.5 Scalar-Vector Operations
A vector can be scaled by a single number. This is exactly how we apply an inflation adjustment or convert a whole price vector to another currency: every element is multiplied by the same scalar.

In [ ]:
inflation_rate = 0.03  # 3% inflation
p_adjusted = p * (1 + inflation_rate)
print(f"p_adjusted = p * (1 + inflation_rate) : {p_adjusted}")

usd_to_eur = 0.92
p_in_eur = p * usd_to_eur
print(f"p_in_eur = p * usd_to_eur             : {p_in_eur}")

<a name="toc_3.4.6"></a>
### 3.4.6 Vector-Vector Dot Product
The dot product is one of the most common operations in economics: it computes **total expenditure** (equivalently, total revenue, or one component of GDP) from a price vector and a quantity vector:
$$ \text{Expenditure} = \sum_{i=0}^{n-1} p_i q_i $$
Let's implement this ourselves with a `for` loop, then compare it with `np.dot`.

In [ ]:
def my_dot(a, b):
    """
    Compute the dot product of two vectors — here, total expenditure across a basket of goods.

    Args:
      a (ndarray (n,)): input vector, e.g. prices
      b (ndarray (n,)): input vector with the same dimension as a, e.g. quantities

    Returns:
      x (scalar): total expenditure sum(a_i * b_i)
    """
    x = 0
    for i in range(a.shape[0]):
        x = x + a[i] * b[i]
    return x

In [ ]:
total_expenditure = my_dot(p, q)
print(f"my_dot(p, q) = {total_expenditure}   (total household expenditure, $ thousand)")

Now let's confirm this against `np.dot`:

In [ ]:
c = np.dot(p, q)
print(f"np.dot(p, q) = {c}, np.dot(p, q).shape = {c.shape}  (scalar)")
c = np.dot(q, p)
print(f"np.dot(q, p) = {c}, np.dot(q, p).shape = {c.shape}  (dot product is commutative)")

<a name="toc_3.4.7"></a>
### 3.4.7 The Need for Speed: vector vs. for loop
Real economic datasets are rarely five goods — think household expenditure surveys with millions of transactions, or high-frequency price data. Let's simulate 10 million (price, quantity) pairs and compare the vectorized `np.dot` against our loop-based `my_dot`.

In [ ]:
np.random.seed(1)
p_large = np.random.rand(10_000_000) * 100   # simulated transaction prices
q_large = np.random.rand(10_000_000) * 10    # simulated transaction quantities

tic = time.time()
c = np.dot(p_large, q_large)
toc = time.time()
print(f"np.dot(p_large, q_large) = {c:.4f}")
print(f"Vectorized version duration: {1000*(toc-tic):.4f} ms")

tic = time.time()
c = my_dot(p_large, q_large)
toc = time.time()
print(f"my_dot(p_large, q_large) = {c:.4f}")
print(f"Loop version duration: {1000*(toc-tic):.4f} ms")

del(p_large); del(q_large)  # free memory

Vectorization provides a large speedup because NumPy exploits data parallelism (SIMD) in the underlying hardware. This matters in economics whenever you work with large panels, high-frequency financial data, or agent-based / Monte Carlo simulations with many replications.

<a name="toc_3.4.8"></a>
### 3.4.8 Vector Operations in Demand Estimation
Vector-vector operations appear constantly in applied econometrics. Suppose we've estimated a simple linear demand model for a good:
$$ \hat{q} = w_0 \cdot \text{price} + w_1 \cdot \text{income} + w_2 \cdot \text{advertising} + b $$
This is exactly a dot product between a coefficient vector $\mathbf{w}$ and a feature vector $\mathbf{x}$, plus an intercept $b$. If our data were stored in a matrix `X_train` of shape (m, n) — m observations by n features — then `X_train[i]` extracts a single observation as a 1-D feature vector, and predicting demand for that observation is a vector-vector dot product.

In [ ]:
w = np.array([-0.8, 0.05, 0.02])   # estimated coefficients: price, income, advertising
b = 12.0                            # intercept

X_train = np.array([[10.0, 500.0, 20.0],   # observation 0: price=10, income=500, ad spend=20
                     [12.0, 520.0, 15.0],   # observation 1
                     [ 9.0, 480.0, 25.0]])  # observation 2

x_i = X_train[1]           # extract one observation -> 1-D vector, shape (n,)
q_hat = np.dot(w, x_i) + b

print(f"x_i has shape {x_i.shape}")
print(f"w has shape {w.shape}")
print(f"Predicted demand for observation 1: q_hat = {q_hat:.2f} units")

<a name="toc_4"></a>
# 4 Matrices in Economics: Panel Data & Input-Output Tables
<a name="toc_4.1"></a>
## 4.1 Abstract
A great deal of macroeconomic and cross-country data is naturally a **matrix**: a two-dimensional array with a row index and a column index. In notation, matrices are denoted with bold capital letters, e.g. $\mathbf{X}$. Two very common economic matrices are:
- **Panel data**: rows are countries (or firms, households), columns are time periods — e.g. GDP by country and year
- **Input-output tables**: rows and columns are both economic sectors, and each entry is the value of inputs one sector buys from another

As with vectors, indexing in NumPy runs from 0 to n-1 for both dimensions.

<a name="toc_4.2"></a>
## 4.2 NumPy Arrays
A 2-D NumPy array has a two-dimensional index `[row, column]`. We will use a 2-D array of shape (m, n) to hold a panel of `m` countries observed over `n` years.

<a name="toc_4.3"></a>
## 4.3 Matrix Creation
The same routines that create 1-D vectors create 2-D arrays when given a shape tuple.

In [ ]:
a = np.zeros((4, 6))
print(f"a shape = {a.shape}\na = \n{a}")

Now let's build a real panel: annual GDP (in $ billion, approximate) for four countries over six years.

In [ ]:
countries = ['USA', 'Germany', 'Japan', 'Brazil']
years     = [2020, 2021, 2022, 2023, 2024, 2025]

GDP = np.array([[21000., 23000., 25400., 26900., 27900., 29300.],   # USA
                [ 3900.,  4300.,  4100.,  4300.,  4500.,  4700.],   # Germany
                [ 5000.,  4900.,  4200.,  4400.,  4100.,  4200.],   # Japan
                [ 1400.,  1600.,  1900.,  2100.,  2200.,  2300.]])  # Brazil

print(f"GDP shape = {GDP.shape}")
print(GDP)

<a name="toc_4.4"></a>
## 4.4 Operations on Matrices
<a name="toc_4.4.1"></a>
### 4.4.1 Indexing
Two indices describe `[row, column]`. Access can return a single element, or a row/column.

In [ ]:
# access a single element: Germany's GDP in 2023 (row 1, column 3)
print(f"GDP[1, 3] = {GDP[1, 3]}  ({countries[1]}, {years[3]}) -> scalar, shape {GDP[1,3].shape}")

# access an entire row: all of the USA's GDP figures
print(f"GDP[0] = {GDP[0]}, shape = {GDP[0].shape}  -> a 1-D vector")

Accessing a matrix by specifying only the row returns a *1-D vector* — the same pattern used with `X[i]` above.

<a name="toc_4.4.2"></a>
### 4.4.2 Slicing
Slicing works the same way on each dimension, separated by a comma.

In [ ]:
# GDP for all countries, for the years 2022-2024 (columns 2 through 4)
print("GDP[:, 2:5] =\n", GDP[:, 2:5], "\nshape:", GDP[:, 2:5].shape, "-> a 2-D array")

# Japan's full GDP series (row 2, all columns)
print("\nGDP[2, :] =", GDP[2, :], ", shape:", GDP[2, :].shape, "-> a 1-D array")

# every value
print("\nGDP[:, :] shape:", GDP[:, :].shape)

<a name="toc_4.5"></a>
## 4.5 Bonus: Matrix-Vector Products in Input-Output Analysis
Wassily Leontief's input-output model represents an economy with a **technical coefficient matrix** $\mathbf{A}$, where $A_{ij}$ is the value of sector $i$'s output needed to produce one unit of sector $j$'s output. Given a **final demand vector** $\mathbf{d}$, the *intermediate* demand each sector must additionally produce is the matrix-vector product $\mathbf{A}\mathbf{d}$ — a single vectorized operation standing in for what would otherwise be a nested loop over every sector pair.

In [ ]:
sectors = ['Agriculture', 'Manufacturing', 'Services']

A = np.array([[0.20, 0.10, 0.05],
              [0.15, 0.25, 0.10],
              [0.10, 0.05, 0.20]])   # technical coefficients

d = np.array([100., 150., 200.])     # final consumer demand by sector

intermediate_demand = np.dot(A, d)
total_output_first_round = d + intermediate_demand

for s, val in zip(sectors, intermediate_demand):
    print(f"Intermediate demand on {s:13s}: {val:.2f}")

print(f"\nTotal first-round output estimate: {total_output_first_round}")

This is only the *first round* of intermediate demand (goods needed to produce goods needed to produce...). The full equilibrium solution requires solving the linear system $\mathbf{x} = (\mathbf{I}-\mathbf{A})^{-1}\mathbf{d}$, using `np.linalg.solve` — a natural next step once you are comfortable with vectors and matrices.

<a name="toc_5"></a>
## Congratulations!
In this lab you used NumPy vectors and matrices to represent and analyze economic data: consumer expenditure, inflation adjustment, demand prediction, cross-country GDP panels, and a first look at input-output analysis — and you confirmed, empirically, why vectorized code matters for economic datasets at scale.